# **EDA on student data**

In [ ]:
# Setup

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
import joblib as jlb 

In [14]:
students = pd.read_csv('../data/StudentsPerformance.csv')

**Data Analysis**

In [15]:
display(students.sample(5))

display(students.info())

display(students.describe())

display(students.isnull().sum())

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
313,female,group C,associate's degree,free/reduced,completed,68,67,69
34,male,group E,some college,standard,none,97,87,82
754,male,group C,associate's degree,free/reduced,none,58,55,53
964,male,group D,some college,standard,none,72,57,58
962,female,group E,associate's degree,standard,none,100,100,100


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


None

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

**Let's get some viz out there**

In [16]:
# Average Gender Intelligence: Correlation between gender and scores
gender_scores = students.groupby('gender').mean(numeric_only=True)
display(gender_scores)
fig = go.Figure(data=[
    go.Bar(name='Math Score', x=gender_scores.index, y= gender_scores['math score']),
    go.Bar(name='Reading Score', x=gender_scores.index, y= gender_scores['reading score']),
    go.Bar(name='Writing Score', x=gender_scores.index, y= gender_scores['writing score'])
])
fig.update_layout(barmode='group', title='Average Scores by Gender', xaxis_title='Gender', yaxis_title='Average Score')
fig.show()

,math score,reading score,writing score
gender,,,
female,63.633205,72.608108,72.467181
male,68.728216,65.473029,63.311203


#### **On average the males are better at math but, the females have more understanding in reading, writing and literature. Highlighting, the differences between each gender's mode of thinking**

In [17]:
race_scores = students.groupby('race/ethnicity').mean(numeric_only=True)
display(race_scores)
fig = go.Figure(data=[
    go.Bar(name='Math Score', x=race_scores.index, y= race_scores['math score']),
    go.Bar(name='Reading Score', x=race_scores.index, y= race_scores['reading score']),
    go.Bar(name='Writing Score', x=race_scores.index, y= race_scores['writing score'])
])
fig.update_layout(barmode='group', title='Average Scores by Race/Ethnicity', xaxis_title='Race/Ethnicity', yaxis_title='Average Score')
fig.show()

,math score,reading score,writing score
race/ethnicity,,,
group A,61.629213,64.674157,62.674157
group B,63.452632,67.352632,65.600000
group C,64.463950,69.103448,67.827586
group D,67.362595,70.030534,70.145038
group E,73.821429,73.028571,71.407143


#### **Group E has the highest grades on average, from this we can also denote that Math is by far the lowest scored subject on average with the exception in group E. So it can be determined that group E has both the highest scores and highest proficiency in Math**

In [18]:
students.head()

# Lets create more insiightful visualizations to understand the data better with a focus on scores using subplots

# Score Distributions
fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Math Score Distribution", "Reading Score Distribution", "Writing Score Distribution"))
fig.add_trace(go.Histogram(x=students['math score'], nbinsx=20, name='Math Score'), row=1, col=1)
fig.add_trace(go.Histogram(x=students['reading score'], nbinsx=20, name='Reading Score'), row=1, col=2)
fig.add_trace(go.Histogram(x=students['writing score'], nbinsx=20, name='Writing Score'), row=1, col=3)

fig.update_layout(title_text='Score Distributions', showlegend=False)
fig.show()

# Scatter Matrix to see relationships between scores
fig = px.scatter_matrix(students, dimensions=['math score', 'reading score', 'writing score'], color='gender', title='Scatter Matrix of Scores by Gender')
fig.update_traces(diagonal_visible=False)
fig.show()

# Categorical Features vs Scores
fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Parental education Vs Grades", "Test Preparation Course Vs Grades", "Lunch Type Vs Grades"))
fig.add_trace(go.Box(x=students['parental level of education'], y=students['math score'], name='Math Score'), row=1, col=1)
fig.add_trace(go.Box(x=students['test preparation course'], y=students['reading score'], name='Reading Score'), row=1, col=2);
fig.add_trace(go.Box(x=students['lunch'], y=students['writing score'], name='Writing Score'), row=1, col=3);

fig.update_layout(title_text='Categorical Features vs Scores')
fig.show()


#### **There is a lot to be said about how factors affect scores but some things are apparent**
- Parents with a masters, have higher standards to their kids grades
- Kids who complete the prep test are **actually prepared** for the exam..... who would have thunk it?
- students with reduced or free meals are actually lower on average to those with standard meals. Food makes a student complete, never forget that

**I smell ML usability, lets set some features and let me convert all the categorical values to numeric**

In [19]:
X = students.drop(columns=['math score', 'reading score', 'writing score'])
y = students[['math score', 'reading score', 'writing score']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

categorical_features = X.columns

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

training time

In [20]:
students_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=69))
])

students_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [21]:
y_pred = students_model.predict(X_test)

In [22]:
# Test R-squared Score for each target variable
r2_scores = [r2_score(y_test.iloc[:, i], y_pred[:, i]) for i in range(y.shape[1])]

print("\nEvaluation (R-squared Score per Target)")
for i, name in enumerate(y.columns):
    print(f"R^2 for {name}: {r2_scores[i]:.4f}")


print("\nPrediction Comparison (True vs. Predicted Scores for the first 5 samples)")
predictions_df = pd.DataFrame(y_pred[:5].round(1), columns=y.columns).add_prefix('Predicted_')
true_df = y_test.head().reset_index(drop=True).add_prefix('True_')
comparison_df = pd.concat([true_df, predictions_df], axis=1)

print(comparison_df.to_markdown(index=False))


Evaluation (R-squared Score per Target)
R^2 for math score: 0.0642
R^2 for reading score: -0.0284
R^2 for writing score: 0.1514

Prediction Comparison (True vs. Predicted Scores for the first 5 samples)
|   True_math score |   True_reading score |   True_writing score |   Predicted_math score |   Predicted_reading score |   Predicted_writing score |
|------------------:|---------------------:|---------------------:|-----------------------:|--------------------------:|--------------------------:|
|                71 |                   87 |                   82 |                   53.6 |                      68.3 |                      63.7 |
|                77 |                   86 |                   86 |                   60.7 |                      71.1 |                      70.3 |
|                85 |                   92 |                   85 |                   69.5 |                      73.4 |                      75.6 |
|                77 |                   90 |       

In [23]:
# Prediction testing
def predict_student_scores(student_conditions: dict, model: Pipeline) -> pd.DataFrame:
    """
    Predicts math, reading, and writing scores for a single student
    using the trained multi-output regression pipeline.

    Args:
        student_conditions (dict): Dictionary of student features.
        model (Pipeline): The trained scikit-learn Pipeline.

    Returns:
        pd.DataFrame: A DataFrame containing the predicted scores.
    """
    new_data = pd.DataFrame([student_conditions])

    prediction_array = model.predict(new_data)

    predicted_scores = pd.DataFrame(
        prediction_array.round(1),
        columns=['Predicted_math score', 'Predicted_reading score', 'Predicted_writing score']
    )
    
    return predicted_scores

# example
new_student = {
    'gender': 'female',
    'race/ethnicity': 'group A',
    'parental level of education': "master's degree",
    'lunch': 'standard',
    'test preparation course': 'completed'
}

print("Prediction for a New Student")
print(pd.Series(new_student).to_markdown(numalign="left", stralign="left"))

predicted_scores_df = predict_student_scores(new_student, students_model)
print("\nPredicted Scores:")
print(predicted_scores_df.to_markdown(index=False, numalign="left", stralign="left"))

Prediction for a New Student
|                             | 0               |
|:----------------------------|:----------------|
| gender                      | female          |
| race/ethnicity              | group A         |
| parental level of education | master's degree |
| lunch                       | standard        |
| test preparation course     | completed       |

Predicted Scores:
| Predicted_math score   | Predicted_reading score   | Predicted_writing score   |
|:-----------------------|:--------------------------|:--------------------------|
| 75.6                   | 83.2                      | 82.6                      |


##### The model performance isnt great but acceptable for a first try. Further hyperparameter tuning and model experimentation can improve results.


In [24]:
# Save the model
jlb.dump(students_model, '../models/students_performance_model.joblib')

['../models/students_performance_model.joblib']